# 02 μP 为什么能让不同宽度的超参数更可迁移？

## 面试回答主线

μP（maximal update parameterization）讨论的是宽度变化时，初始化、学习率和参数尺度如何一起设定，使特征和函数更新保持可比。它不是“把所有层都除以宽度”的口号：输入层、隐藏层和读出层有不同的缩放规则。面试时应解释目标是最大更新而非单纯方差恒定，并说明要先在小宽度调参、再迁移到大宽度。本实验以工单风险分类为例，测量 8 宽与 64 宽网络在一次更新前后的 logit 漂移。数值接近只说明该受控初始化的一致性，不等价于大模型训练收益。

**核心公式：** 两层网络可写为 $h=\phi(xW_1)$、$f=hW_2$。普通参数化常令 $W_2=O(1/\sqrt{n})$；μP 为保持宽度增大时函数更新量可控，会为不同参数张量配置不同的初始化和学习率尺度。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
def standard_drift(width):  # 测量普通 Xavier 风格宽度变化后的单步函数漂移。
    torch.manual_seed(width)  # 为每个宽度固定可复现实验种子。
    w1 = torch.randn(3, width, requires_grad=True) / math.sqrt(3.0)  # 创建普通输入矩阵。
    w2 = torch.randn(width, 2, requires_grad=True) / math.sqrt(width)  # 创建普通读出矩阵。
    before = torch.tanh(features @ w1) @ w2  # 记录更新前 logits。
    loss = torch.nn.functional.cross_entropy(before, labels)  # 计算分类损失。
    grad_w1, grad_w2 = torch.autograd.grad(loss, [w1, w2])  # 获得两类参数的梯度。
    after = torch.tanh(features @ (w1 - 0.08 * grad_w1)) @ (w2 - 0.08 * grad_w2)  # 使用同一普通学习率模拟更新。
    return float((after - before).pow(2).mean().sqrt())  # 返回 logit 的 RMS 漂移。
standard_small = standard_drift(8)  # 计算窄网络普通参数化漂移。
standard_wide = standard_drift(64)  # 计算宽网络普通参数化漂移。
baseline_metric = abs(standard_wide - standard_small)  # 用跨宽度漂移差作为基线指标。
print(f'普通参数化：width=8 漂移={standard_small:.4f}，width=64 漂移={standard_wide:.4f}')  # 展示基线对照。


普通参数化：width=8 漂移=0.0278，width=64 漂移=0.2115


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
class MuPClassifier(nn.Module):  # 用显式参数矩阵实现一个可观察宽度尺度的分类器。
    def __init__(self, width):  # 接收隐藏宽度并建立对应参数。
        super().__init__()  # 初始化 PyTorch 模块父类。
        self.width = width  # 保存宽度供学习率分组使用。
        self.w1 = nn.Parameter(torch.randn(3, width) / math.sqrt(3.0))  # 初始化输入到隐藏的矩阵。
        self.w2 = nn.Parameter(torch.randn(width, 2) / math.sqrt(float(width)))  # 使用宽度稳定的读出初始化以隔离更新尺度。
    def forward(self, batch):  # 明确写出前向传播而不是调用现成网络。
        hidden = torch.tanh(batch @ self.w1)  # 计算隐藏特征。
        return hidden @ self.w2  # 返回读出 logits。
def mup_drift(width):  # 测量 μP 风格分组更新的函数漂移。
    torch.manual_seed(width)  # 固定宽度实验的随机性。
    model = MuPClassifier(width)  # 建立指定宽度的手写网络。
    before = model(features).detach()  # 保存更新前函数输出。
    loss = torch.nn.functional.cross_entropy(model(features), labels)  # 计算当前损失。
    grad_w1, grad_w2 = torch.autograd.grad(loss, [model.w1, model.w2])  # 分别取得两组梯度。
    with torch.no_grad():  # 手写参数更新无需构建二阶图。
        model.w1 -= 0.08 * grad_w1  # 对输入矩阵使用基础学习率。
        model.w2 -= 0.08 * grad_w2 / float(width)  # 对读出矩阵使用宽度相关学习率以保持函数更新可比。
    after = model(features).detach()  # 获取更新后函数输出。
    return float((after - before).pow(2).mean().sqrt()), float(loss)  # 返回漂移和损失供表格解释。
mup_small, mup_small_loss = mup_drift(8)  # 计算窄网络 μP 指标。
mup_wide, mup_wide_loss = mup_drift(64)  # 计算宽网络 μP 指标。
core_metric = abs(mup_wide - mup_small)  # 计算 μP 跨宽度函数漂移差。
print(f'μP 参数化：width=8 漂移={mup_small:.4f}，width=64 漂移={mup_wide:.4f}，差值={core_metric:.4f}')  # 展示核心对照。


μP 参数化：width=8 漂移=0.0156，width=64 漂移=0.0167，差值=0.0011


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=0.183670
核心机制     | 指标=0.001062


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **μP** 的关键状态与更新路径。生产中需要明确 base shape、参数类别和 checkpoint 转换；只复制学习率而不复制 parametrization，迁移没有理论意义。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
broken_model = MuPClassifier(64)  # 创建宽网络作为错误缩放示例。
broken_before = broken_model(features).detach()  # 保存错误更新前 logits。
broken_loss = torch.nn.functional.cross_entropy(broken_model(features), labels)  # 计算错误示例的损失。
broken_g1, broken_g2 = torch.autograd.grad(broken_loss, [broken_model.w1, broken_model.w2])  # 获取错误示例梯度。
with torch.no_grad():  # 开始错误更新。
    broken_model.w1 -= 0.08 * broken_g1  # 正确更新输入矩阵以隔离错误来源。
    broken_model.w2 -= 0.08 * 64.0 * 64.0 * broken_g2  # 故意把读出学习率额外放大一个宽度因子。
failure_metric = float((broken_model(features) - broken_before).pow(2).mean().sqrt())  # 记录错误缩放导致的漂移。
fixed_model = MuPClassifier(64)  # 创建修复后的宽网络。
fixed_before = fixed_model(features).detach()  # 保存修复前 logits。
fixed_loss = torch.nn.functional.cross_entropy(fixed_model(features), labels)  # 计算修复示例损失。
fixed_g1, fixed_g2 = torch.autograd.grad(fixed_loss, [fixed_model.w1, fixed_model.w2])  # 获取修复梯度。
with torch.no_grad():  # 开始修复更新。
    fixed_model.w1 -= 0.08 * fixed_g1  # 更新输入矩阵。
    fixed_model.w2 -= 0.08 * fixed_g2 / 64.0  # 恢复按宽度缩小的读出学习率。
fix_metric = float((fixed_model(features) - fixed_before).pow(2).mean().sqrt())  # 记录正确尺度的漂移。
print(f'失败：读出层错误放大学习率后漂移={failure_metric:.3f}；修复后漂移={fix_metric:.3f}')  # 展示尺度错误与修复。


失败：读出层错误放大学习率后漂移=774.902；修复后漂移=0.014


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产中需要明确 base shape、参数类别和 checkpoint 转换；只复制学习率而不复制 parametrization，迁移没有理论意义。

**常见坑：** 仅比较初始 loss，或把输出层与隐藏层用同一学习率缩放，都会掩盖真正的函数更新差异。

**延伸追问：** 如何在 attention 的 QKV、输出投影、embedding 和 MoE gate 上定义参数组？张量并行后 base shape 应按全局还是分片形状？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert standard_small > 0.0  # 验证普通参数化产生了函数更新。
assert mup_small > 0.0  # 验证 μP 参数化产生了函数更新。
assert failure_metric > fix_metric  # 验证错误的二次宽度缩放会放大函数漂移。
assert mup_small_loss > 0.0  # 验证实验使用了真实的交叉熵目标。
